In [ ]:
# --- Standard library ---
import os          # reading environment variables, file paths
import re           # regex, used for detecting section headings in extracted text
import json         # saving intermediate chunk/metadata objects to disk
import subprocess   # calling command-line tools (pdfinfo, pdffonts, pdftoppm) from Python
import base64       # encoding images to base64 for the Claude vision API and Voyage multimodal API
from pathlib import Path
import time

In [ ]:
# --- Third-party libraries ---
from dotenv import load_dotenv   # python-dotenv: loads key=value pairs from .env into os.environ
import pymupdf                     # PyMuPDF, imported under its real name (not the unrelated "fitz" PyPI package)
from PIL import Image             # Pillow, used to open/resize/save extracted and rasterized images
import io

from anthropic import Anthropic   # Official Anthropic SDK, used directly for vision calls (router + image classification)
import voyageai                   # Voyage AI client, used for multimodal text+image embeddings

import chromadb                   # ChromaDB client, our vector store for text and image embeddings
from rank_bm25 import BM25Okapi   # BM25 term-based retrieval algorithm implementation

from langchain_anthropic import ChatAnthropic              # LangChain wrapper around Claude, needed by LLMGraphTransformer
         # LangChain's standard document object (text + metadata)

In [ ]:
# --- Load environment variables from .env ---
load_dotenv()

ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
VOYAGE_API_KEY = os.environ["VOYAGE_API_KEY"]
# COHERE_API_KEY and LANGCHAIN_* vars are used in retrieval_pipeline.ipynb, not needed for ingestion

In [ ]:
# --- Initialize clients ---
anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)      # used for Haiku vision calls (classification + figure extraction)
voyage_client = voyageai.Client(api_key=VOYAGE_API_KEY)       # used for voyage-multimodal-3 embeddings

In [ ]:
# ChromaDB persistent client — data survives between notebook runs, stored on disk
chroma_client = chromadb.PersistentClient(path="./data/chroma_db")
text_collection = chroma_client.get_or_create_collection(name="text_chunks")
image_collection = chroma_client.get_or_create_collection(name="image_chunks")

In [ ]:
print("Setup complete. Clients initialized: Anthropic, Voyage, ChromaDB (2 collections).")

In [ ]:
# Find PDFs in data/raw_filings/ and parse exchange + ticker + year out of each filename

RAW_FILINGS_DIR = Path("data/raw_filings")

# Regex to parse "NASDAQ_MSFT_2023.pdf" into exchange="NASDAQ", ticker="MSFT", year="2023"
# (matches the actual filenames downloaded: EXCHANGE_TICKER_YEAR.pdf)
FILENAME_PATTERN = re.compile(r"^([A-Z]+)_([A-Z]+)_(\d{4})\.pdf$")

pdf_files = sorted(RAW_FILINGS_DIR.glob("*.pdf"))

documents_to_process = []  # list of dicts: {path, exchange, ticker, year}

for pdf_path in pdf_files:
    match = FILENAME_PATTERN.match(pdf_path.name)
    if not match:
        print(f"WARNING: '{pdf_path.name}' does not match EXCHANGE_TICKER_YEAR.pdf, skipping.")
        continue
    exchange, ticker, year = match.groups()
    documents_to_process.append({
        "path": pdf_path,
        "exchange": exchange,
        "ticker": ticker,
        "year": int(year),
    })

print(f"Found {len(documents_to_process)} valid PDFs:")
for d in documents_to_process:
    print(f"  {d['exchange']} {d['ticker']} {d['year']}  ->  {d['path'].name}")

expected_tickers = {"MSFT", "NVDA", "KO", "TGT"}
found_tickers = {d["ticker"] for d in documents_to_process}
if found_tickers != expected_tickers:
    print(f"\nWARNING: expected tickers {expected_tickers}, found {found_tickers}")

# Check that every ticker has the same set of fiscal years covered.
# This matters for cross-company comparison queries later (e.g. "compare risk
# factors across all four companies in the same year") -- if one company's
# years don't line up with the rest, that comparison silently breaks.
years_by_ticker = {}
for d in documents_to_process:
    years_by_ticker.setdefault(d["ticker"], set()).add(d["year"])

all_year_sets = list(years_by_ticker.values())
if len(set(frozenset(y) for y in all_year_sets)) > 1:
    print("\nWARNING: not all companies cover the same fiscal years:")
    for ticker, years in sorted(years_by_ticker.items()):
        print(f"  {ticker}: {sorted(years)}")
    print("  Cross-company comparisons for a mismatched year will have incomplete data.")

In [ ]:
# Run pdfinfo/pdffonts/pdfimages on each PDF to check page count, text layer, raster image count

def run_cli(args):
    """Run a command-line tool and return its stdout as text.
    Used for the poppler-utils tools (pdfinfo, pdffonts, pdfimages, pdftoppm),
    which are system binaries, not Python libraries, so we call them via subprocess."""
    result = subprocess.run(args, capture_output=True, text=True)
    return result.stdout


def content_inventory(pdf_path):
    """Quick diagnostic on one PDF: page count, font status, raster image count."""
    info = run_cli(["pdfinfo", str(pdf_path)])
    page_count = 0
    for line in info.splitlines():
        if line.startswith("Pages:"):
            page_count = int(line.split(":")[1].strip())

    fonts_output = run_cli(["pdffonts", str(pdf_path)])
    # pdffonts prints a header + separator even when there are zero fonts,
    # so more than 2 lines means at least one font exists (a real text layer).
    has_text_layer = len(fonts_output.strip().splitlines()) > 2

    images_output = run_cli(["pdfimages", "-list", str(pdf_path)])
    raster_image_count = max(0, len(images_output.strip().splitlines()) - 2)

    return {
        "page_count": page_count,
        "has_text_layer": has_text_layer,
        "raster_image_count": raster_image_count,
    }


for doc in documents_to_process:
    inventory = content_inventory(doc["path"])
    doc.update(inventory)
    flag = "" if inventory["has_text_layer"] else "  <-- NO TEXT LAYER, may be scanned!"
    print(f"{doc['ticker']} {doc['year']}: {inventory['page_count']} pages, "
          f"{inventory['raster_image_count']} raster images{flag}")

In [ ]:
# Use PyMuPDF to pull per-page text and embedded raster images out of each PDF

EXTRACTED_TEXT_DIR = Path("data/extracted_text")
EXTRACTED_IMAGES_DIR = Path("data/extracted_images")
EXTRACTED_TEXT_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_IMAGES_DIR.mkdir(parents=True, exist_ok=True)


def extract_text_and_images(doc_meta):
    """Extract per-page text and raster images from one PDF using PyMuPDF.

    Returns:
        page_texts: list of strings, one per page
        raster_images: list of dicts {page_num, image_bytes, image_path}
    """
    pdf = pymupdf.open(doc_meta["path"])  # pymupdf.open reads the PDF into memory
    page_texts = []
    raster_images = []

    for page_num, page in enumerate(pdf):
        # get_text() pulls the text layer for this page, in reading order
        page_texts.append(page.get_text())

        # get_images() lists raster image objects embedded on this page
        # (does NOT catch vector-drawn charts — see Step 5)
        for img_index, img in enumerate(page.get_images(full=True)):
            xref = img[0]  # xref is the PDF internal object reference for this image
            pix = pymupdf.Pixmap(pdf, xref)  # Pixmap = the actual pixel data for the image
            if pix.n - pix.alpha > 3:  # CMYK or other non-RGB colorspace
                pix = pymupdf.Pixmap(pymupdf.csRGB, pix)  # convert to RGB so PIL/Voyage can read it

            image_bytes = pix.tobytes("png")
            image_filename = f"{doc_meta['ticker']}_{doc_meta['year']}_p{page_num}_img{img_index}.png"
            image_path = EXTRACTED_IMAGES_DIR / image_filename
            with open(image_path, "wb") as f:
                f.write(image_bytes)

            raster_images.append({
                "page_num": page_num,
                "image_bytes": image_bytes,
                "image_path": image_path,
            })

    pdf.close()
    return page_texts, raster_images


all_page_texts = {}      # {(ticker, year): [page_text, ...]}
all_raster_images = {}   # {(ticker, year): [image_dict, ...]}

for doc in documents_to_process:
    page_texts, raster_images = extract_text_and_images(doc)
    key = (doc["ticker"], doc["year"])
    all_page_texts[key] = page_texts
    all_raster_images[key] = raster_images

    # Save the raw extracted text to disk for inspection/debugging
    text_out_path = EXTRACTED_TEXT_DIR / f"{doc['ticker']}_{doc['year']}_raw.txt"
    with open(text_out_path, "w", encoding="utf-8") as f:
        f.write("\n\n<<<PAGE BREAK>>>\n\n".join(page_texts))

    print(f"{doc['ticker']} {doc['year']}: extracted {len(page_texts)} pages of text, "
          f"{len(raster_images)} raster images")

In [ ]:
# Score each page's text density AND check for metric-keyword pages missing percentages; rasterize + vision-read flagged pages

RASTERIZED_PAGES_DIR = Path("data/rasterized_pages")
RASTERIZED_PAGES_DIR.mkdir(parents=True, exist_ok=True)

TEXT_DENSITY_THRESHOLD = 300  # characters per page; below this, the page is nearly blank
                                # (catches cover pages, back covers, section divider pages)

# Second trigger, catches the case text-density alone misses: a page with PLENTY of
# narrative text (so it clears the density threshold above) but whose actual growth/
# metric numbers are rendered as vector-graphic callouts, not text. We confirmed this
# directly on the Microsoft filing: page 30 had long descriptive paragraphs for each
# metric (well over 300 characters) but zero "%" signs anywhere, because the growth
# percentages themselves were vector-drawn, not text.
METRIC_KEYWORD_PATTERN = re.compile(r"(?i)\bgrowth\b|\bmetrics\b|\bsubscribers\b|\bseat growth\b")


def page_likely_missing_figures(text):
    """True if this page is near-blank, OR mentions growth/metrics repeatedly
    but contains no percentage signs at all (a strong signal the actual numbers
    are vector graphics, not text)."""
    if len(text) < TEXT_DENSITY_THRESHOLD:
        return True
    keyword_hits = len(METRIC_KEYWORD_PATTERN.findall(text))
    has_percent_sign = "%" in text
    return keyword_hits >= 2 and not has_percent_sign


def rasterize_page(pdf_path, page_num, ticker, year, dpi=200):
    """Rasterize a single PDF page to a JPEG using pdftoppm (Poppler).

    BUG FIX: the prefix now includes page_num itself (not just ticker/year).
    The original version used a shared prefix like "MSFT_2024_page" for every
    page of a document, and pdftoppm's own output numbering isn't zero-padded,
    so once multiple pages had been rasterized for the same document, glob()
    would match ALL of them and sorted(...)[-1] picked whichever filename
    sorted last as a STRING (e.g. "page-91" sorts after "page-30" because '9' > '3'),
    silently returning the wrong page's image. Including page_num in the prefix
    makes each page's output filename unique, so this collision can't happen.

    Also bumped default DPI from 150 to 200: 150 DPI risked being too low to
    read small vector-drawn percentage badges/icons clearly during vision extraction.

    pdftoppm is 1-indexed and appends its own page-numbering suffix, so we
    still glob for the exact output rather than predicting the full filename.
    """
    output_prefix = RASTERIZED_PAGES_DIR / f"{ticker}_{year}_page{page_num}"
    subprocess.run([
        "pdftoppm", "-jpeg", "-r", str(dpi),
        "-f", str(page_num + 1), "-l", str(page_num + 1),  # pdftoppm pages are 1-indexed
        str(pdf_path), str(output_prefix)
    ])
    matches = sorted(RASTERIZED_PAGES_DIR.glob(f"{ticker}_{year}_page{page_num}-*.jpg"))
    return matches[-1] if matches else None


def extract_figures_with_vision(image_path, page_num, ticker, year):
    """Send a rasterized page to Claude Haiku vision and ask it to read out any
    figures/percentages/metrics visible on the page that plain text extraction
    would have missed (vector-drawn charts, callout numbers, etc.)."""
    with open(image_path, "rb") as f:
        image_b64 = base64.standard_b64encode(f.read()).decode("utf-8")

    response = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",  # same Haiku model used for the router later, matches task complexity
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": "image/jpeg", "data": image_b64}},
                {"type": "text", "text": (
                    "This is a page from a company annual report. Look carefully, including inside "
                    "any circular badges, icons, colored shapes, or small graphic callouts, for "
                    "numbers or percentages that may not be plain text. List every specific figure, "
                    "percentage, or metric shown on this page as short factual sentences "
                    "(e.g. 'LinkedIn revenue growth: 10% year over year'). "
                    "If there are truly no numeric figures anywhere on this page, respond with exactly the single word: NONE"
                )},
            ],
        }],
    )
    return response.content[0].text.strip()


def is_none_response(text):
    """Robust check for Haiku's 'nothing found' response. A strict equality check
    (text == "NONE") is fragile -- the model can return "NONE.", "None", trailing
    whitespace, etc., and a strict check silently lets those slip through as if
    they were real figure data. Normalize first, then compare."""
    normalized = text.strip().strip(".").strip().upper()
    return normalized == "NONE"


vision_figure_chunks = []  # list of dicts: {ticker, year, page_num, text}
pages_flagged = 0

for doc in documents_to_process:
    key = (doc["ticker"], doc["year"])
    page_texts = all_page_texts[key]

    for page_num, text in enumerate(page_texts):
        if page_likely_missing_figures(text):
            pages_flagged += 1
            image_path = rasterize_page(doc["path"], page_num, doc["ticker"], doc["year"])
            if image_path is None:
                continue
            figure_text = extract_figures_with_vision(image_path, page_num, doc["ticker"], doc["year"])
            if not is_none_response(figure_text):
                vision_figure_chunks.append({
                    "ticker": doc["ticker"],
                    "year": doc["year"],
                    "page_num": page_num,
                    "text": figure_text,
                })

print(f"Flagged {pages_flagged} pages across all documents (low-density or metric-keyword-without-percent).")
print(f"Of those, {len(vision_figure_chunks)} actually contained real figures once read by vision "
      f"(the rest were correctly filtered out as NONE).")
for chunk in vision_figure_chunks[:5]:
    print(f"  {chunk['ticker']} {chunk['year']} p{chunk['page_num']}: {chunk['text'][:80]}...")

In [ ]:
from collections import Counter

print(f"Total pages flagged: {pages_flagged}")
print(f"Total pages with real extracted figures: {len(vision_figure_chunks)}")

if vision_figure_chunks:
    doc_counts = Counter((c["ticker"], c["year"]) for c in vision_figure_chunks)
    print("\nVision-extracted figures per document:")
    for (ticker, year), count in sorted(doc_counts.items()):
        print(f"  {ticker} {year}: {count}")
else:
    print("  No vision-extracted figures found across any document -- worth spot-checking a few "
          "flagged pages manually to confirm this is correct and not a detection issue.")

In [ ]:
# --- Shared helper: prepare any image for the Claude vision API ---
# Claude's API rejects images over 10 MB. Some raster images extracted straight out of
# a PDF (especially Nvidia's, which run 150-190+ per document) can be large, uncompressed
# embedded photos well over that limit. Resizing to Anthropic's recommended max dimension
# and re-encoding as JPEG fixes the size problem AND keeps vision API costs/tokens down
# (resolution beyond ~1568px on the long side adds cost without adding model accuracy).
def prepare_image_for_vision(image_bytes, max_dimension=1568, jpeg_quality=85):
    """Resize (if needed) and re-encode an image as JPEG so it's safely under
    the 10 MB API limit. Returns (jpeg_bytes, media_type)."""
    img = Image.open(io.BytesIO(image_bytes))
    if img.mode != "RGB":
        img = img.convert("RGB")  # JPEG has no alpha channel; also normalizes CMYK/palette images
    if max(img.size) > max_dimension:
        scale = max_dimension / max(img.size)
        new_size = (int(img.width * scale), int(img.height * scale))
        img = img.resize(new_size, Image.LANCZOS)
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=jpeg_quality)
    return buffer.getvalue(), "image/jpeg"

In [ ]:
# Send every extracted raster image to Haiku vision and keep only chart/table/photo labels

KEEP_LABELS = {"chart", "table", "photo"}
DISCARD_LABELS = {"logo", "decorative"}


def classify_image(image_bytes):
    """Ask Claude Haiku vision to label one extracted image.

    BUG FIX: images are now resized/re-encoded via prepare_image_for_vision()
    before sending. One of Nvidia's raw extracted images was 12.7 MB (uncompressed,
    full resolution), which the API rejected with a 400 error (10 MB limit).
    """
    resized_bytes, media_type = prepare_image_for_vision(image_bytes)
    image_b64 = base64.standard_b64encode(resized_bytes).decode("utf-8")

    response = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=20,
        messages=[{
            "role": "user",
            "content": [
                {"type": "image", "source": {"type": "base64", "media_type": media_type, "data": image_b64}},
                {"type": "text", "text": (
                    "Classify this image from a company annual report as exactly one word: "
                    "chart, table, photo, logo, or decorative. Respond with only that one word."
                )},
            ],
        }],
    )
    label = response.content[0].text.strip().lower()
    return label if label in KEEP_LABELS | DISCARD_LABELS else "decorative"  # safe default


kept_images = []  # list of dicts: {ticker, year, page_num, image_path, label}

for doc in documents_to_process:
    key = (doc["ticker"], doc["year"])
    for img in all_raster_images[key]:
        label = classify_image(img["image_bytes"])
        if label in KEEP_LABELS:
            kept_images.append({
                "ticker": doc["ticker"],
                "year": doc["year"],
                "page_num": img["page_num"],
                "image_path": img["image_path"],
                "label": label,
            })

print(f"Classified all raster images. Kept {len(kept_images)} as chart/table/photo, "
      f"discarded the rest as logo/decorative.")

In [ ]:
# Split each document into sections by heading, then into 500-char chunks with 100-char overlap

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # characters per chunk, tuned to roughly one to two sentences
    chunk_overlap=100,  # 20% overlap, protects against a key sentence being cut at a boundary
    separators=["\n\n", "\n", ". ", " ", ""],  # tries paragraph, then line, then sentence, then word
)


# BUG FIX: the original approach hardcoded a fixed list of expected heading phrases
# (e.g. "business overview", "risk factors"). Real annual reports don't share one
# vocabulary -- Microsoft's actual headings are things like "Industry Trends" and
# "More Personal Computing", which don't match any guessed phrase, so every chunk
# was falling back to "full_document" and section-level metadata was lost entirely.
#
# Fix: detect headings by SHAPE instead of wording. Annual report headings share a
# visual pattern across companies regardless of vocabulary: short, standalone lines,
# Title Case or ALL CAPS, no trailing sentence punctuation. This generalizes across
# all 4 companies without needing to guess each one's specific section names.
def is_likely_heading(line):
    """True if this line looks like a section heading rather than a normal sentence."""
    line = line.strip()
    if not line or len(line) > 80:
        return False
    if line.endswith((".", ",", ";", ":")):  # normal sentences end in punctuation, headings don't
        return False
    words = line.split()
    if len(words) > 10:  # headings are short; long lines are body text even if capitalized
        return False
    if line.isupper():  # e.g. "SUMMARY RESULTS OF OPERATIONS"
        return True
    # Title Case check: most words start with a capital letter (allows small
    # connector words like "and"/"of" to stay lowercase, as real headings do)
    capitalized = sum(1 for w in words if w[:1].isupper())
    return capitalized / len(words) >= 0.6


def slugify_heading(heading_text):
    """Turn a heading like 'Industry Trends' into a clean metadata value: 'industry_trends'."""
    slug = re.sub(r"[^a-z0-9]+", "_", heading_text.lower()).strip("_")
    return slug[:60] if slug else "section"


def split_into_sections(full_text):
    """Stage 1: break the full document text into (section_name, section_text) pairs
    by scanning line by line for heading-shaped lines. Text before the first
    detected heading is labeled 'front_matter'."""
    lines = full_text.split("\n")
    heading_positions = []  # list of (char_offset_in_full_text, heading_text)
    offset = 0
    for line in lines:
        if is_likely_heading(line):
            heading_positions.append((offset, line.strip()))
        offset += len(line) + 1  # +1 accounts for the "\n" that split() removed

    if not heading_positions:
        return [("full_document", full_text)]  # fallback if truly no headings found

    sections = []
    front_matter = full_text[:heading_positions[0][0]].strip()
    if front_matter:
        sections.append(("front_matter", front_matter))

    for i, (start_offset, heading_text) in enumerate(heading_positions):
        section_name = slugify_heading(heading_text)
        content_start = start_offset + len(heading_text)
        content_end = heading_positions[i + 1][0] if i + 1 < len(heading_positions) else len(full_text)
        sections.append((section_name, full_text[content_start:content_end]))

    return sections


all_chunks = []  # list of dicts: {chunk_id, text, ticker, year, section}
chunk_counter = 0

for doc in documents_to_process:
    key = (doc["ticker"], doc["year"])
    full_text = "\n\n".join(all_page_texts[key])
    sections = split_into_sections(full_text)

    for section_name, section_text in sections:
        # Stage 2: character-level split within this section
        pieces = splitter.split_text(section_text)
        for piece in pieces:
            chunk_id = f"{doc['ticker']}_{doc['year']}_chunk_{chunk_counter}"
            chunk_counter += 1
            all_chunks.append({
                "chunk_id": chunk_id,
                "text": piece,
                "ticker": doc["ticker"],
                "year": doc["year"],
                "section": section_name,
                "source": "pdf_text",
            })

# Add the vision-extracted figure chunks from Step 5 as their own chunks
for fig in vision_figure_chunks:
    chunk_id = f"{fig['ticker']}_{fig['year']}_chunk_{chunk_counter}"
    chunk_counter += 1
    all_chunks.append({
        "chunk_id": chunk_id,
        "text": fig["text"],
        "ticker": fig["ticker"],
        "year": fig["year"],
        "section": f"page_{fig['page_num']}_figures",
        "source": "vision_figure_extraction",
    })

print(f"Built {len(all_chunks)} total chunks across all 8 documents "
      f"({len(vision_figure_chunks)} from vision figure extraction).")

# Save chunks to disk so retrieval_pipeline.ipynb can load them without re-running ingestion
with open("data/chunks.json", "w", encoding="utf-8") as f:
    json.dump(all_chunks, f, indent=2)

In [ ]:
from collections import Counter

section_counts = Counter(c["section"] for c in all_chunks)
full_document_count = section_counts.get("full_document", 0)
full_document_pct = 100 * full_document_count / len(all_chunks) if all_chunks else 0

print(f"Total chunks: {len(all_chunks)}")
print(f"Distinct sections detected: {len(section_counts)}")
print(f"Chunks still falling back to 'full_document': {full_document_count} ({full_document_pct:.1f}%)")
if full_document_pct > 50:
    print("  WARNING: most chunks have no real section detected -- heading detection may not be working "
          "for this document set, check is_likely_heading() against a sample of the actual extracted text.")
else:
    print("  OK: section detection is finding real headings.")

print("\nTop 10 most common sections:")
for section, count in section_counts.most_common(10):
    print(f"  {section}: {count} chunks")

In [ ]:
# Tokenize all chunks and build a BM25 index for term-based retrieval

def simple_tokenize(text):
    """Lowercase, alphanumeric-only tokenization for BM25.
    BM25 works on token overlap, so this doesn't need to be fancy —
    just consistent between indexing and querying."""
    return re.findall(r"[a-z0-9]+", text.lower())


tokenized_corpus = [simple_tokenize(chunk["text"]) for chunk in all_chunks]
bm25_index = BM25Okapi(tokenized_corpus)  # builds term-frequency statistics over all chunks

print(f"BM25 index built over {len(tokenized_corpus)} chunks.")

# Quick sanity check: does a keyword query return something plausible?
sample_query = simple_tokenize("revenue growth")
scores = bm25_index.get_scores(sample_query)
top_idx = scores.argsort()[-3:][::-1]
print("\nTop 3 BM25 matches for 'revenue growth':")
for i in top_idx:
    c = all_chunks[i]
    print(f"  [{c['ticker']} {c['year']} / {c['section']}] {c['text'][:100]}...")

In [ ]:
# Embed all text chunks with Voyage (batched) and store them in the ChromaDB text collection

def embed_texts_batch(texts, batch_size=50):
    """Embed a list of texts with Voyage in batches (cheaper and faster than one call per text).
    input_type='document' tells Voyage these are things being indexed, not a query —
    Voyage optimizes the embedding differently for documents vs. queries.

    BUG FIX: voyage-multimodal-3 is only usable through multimodal_embed(), not the
    plain embed() method (that method only accepts text-only models like voyage-3).
    Since we specifically chose voyage-multimodal-3 so text and image embeddings share
    the same vector space, text has to go through multimodal_embed() too, just with
    text-only inputs. multimodal_embed() expects "inputs" as a list of lists, where
    each inner list holds one item's content (a string, an image, or a mix) -- for
    text-only embedding, each chunk gets wrapped as its own single-item list.
    """
    all_embeddings = []
    num_batches = (len(texts) + batch_size - 1) // batch_size

    for batch_num, i in enumerate(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        multimodal_inputs = [[text] for text in batch]  # wrap each text as its own [text] input

        # Retry with backoff as a SAFETY NET for transient failures, not the primary
        # pacing strategy -- see the proactive sleep below for why.
        max_retries = 5
        for attempt in range(max_retries):
            try:
                result = voyage_client.multimodal_embed(
                    inputs=multimodal_inputs, model="voyage-multimodal-3", input_type="document"
                )
                break
            except Exception as e:
                if "RateLimitError" in type(e).__name__ or "rate limit" in str(e).lower():
                    wait_seconds = 20 * (attempt + 1)
                    print(f"  Rate limited, waiting {wait_seconds}s before retry "
                          f"(attempt {attempt + 1}/{max_retries})...")
                    time.sleep(wait_seconds)
                else:
                    raise
        else:
            raise RuntimeError(f"Failed to embed batch starting at index {i} after {max_retries} retries.")

        all_embeddings.extend(result.embeddings)
        print(f"  Embedded batch {batch_num + 1}/{num_batches} "
              f"({min(i + batch_size, len(texts))}/{len(texts)} chunks total)")

        
        if batch_num < num_batches - 1:
            time.sleep(21)

    return all_embeddings


chunk_texts = [c["text"] for c in all_chunks]
chunk_embeddings = embed_texts_batch(chunk_texts)

print(f"Finished embedding {len(chunk_embeddings)} chunks. Ready to store in ChromaDB (next cell).")

In [ ]:
CHROMA_ADD_BATCH_SIZE = 5000

all_chunk_ids = [c["chunk_id"] for c in all_chunks]
all_metadatas = [{
    "ticker": c["ticker"],
    "year": c["year"],
    "section": c["section"],
    "source": c["source"],
} for c in all_chunks]

for i in range(0, len(all_chunks), CHROMA_ADD_BATCH_SIZE):
    end = i + CHROMA_ADD_BATCH_SIZE
    text_collection.add(
        ids=all_chunk_ids[i:end],
        embeddings=chunk_embeddings[i:end],
        documents=chunk_texts[i:end],
        metadatas=all_metadatas[i:end],
    )
    print(f"  Stored chunks {i} to {min(end, len(all_chunks))} in ChromaDB...")

print(f"Embedded and stored {len(all_chunks)} text chunks in ChromaDB 'text_chunks' collection.")

In [ ]:
print(f"all_chunks: {len(all_chunks)}")
print(f"chunk_embeddings: {len(chunk_embeddings)}")
print(f"Vectors actually stored in ChromaDB text_chunks: {text_collection.count()}")
if not (len(all_chunks) == len(chunk_embeddings) == text_collection.count()):
    print("  WARNING: counts don't match across chunks/embeddings/stored -- something was dropped somewhere.")
else:
    print("  OK: all three counts match.")

In [ ]:
# Embed each kept image with Voyage multimodal and store it in the ChromaDB image collection

def embed_image(image_path):
    """Embed a single image with Voyage multimodal, same vector space as the text embeddings above,
    so a chart and a paragraph describing it can both be found by the same query.

    Resized via the same prepare_image_for_vision() helper used in Step 6, since some
    raw extracted images (Nvidia's especially) run well over 10 MB at full resolution --
    keeping embedding input sizes consistent and small avoids similar size-limit issues here."""
    with open(image_path, "rb") as f:
        resized_bytes, _ = prepare_image_for_vision(f.read())
    img = Image.open(io.BytesIO(resized_bytes))
    result = voyage_client.multimodal_embed(
        inputs=[[img]],           # Voyage multimodal API takes a list of "inputs", each itself a list of content
        model="voyage-multimodal-3",
        input_type="document",
    )
    return result.embeddings[0]


image_ids, image_embeddings, image_metadatas = [], [], []

for i, img in enumerate(kept_images):
    image_id = f"{img['ticker']}_{img['year']}_image_{i}"
    embedding = embed_image(img["image_path"])
    image_ids.append(image_id)
    image_embeddings.append(embedding)
    image_metadatas.append({
        "ticker": img["ticker"],
        "year": img["year"],
        "page_num": img["page_num"],
        "label": img["label"],
        "image_path": str(img["image_path"]),
    })

# Same batching safety as text_collection.add() -- Chroma's max batch
# size (5461) applies here too, even though kept_images is unlikely to exceed
# it in this project, batching defensively costs nothing and avoids a repeat
# of the same error if the image count ever grows.
CHROMA_ADD_BATCH_SIZE = 5000

image_documents = [f"{m['label']} from {m['ticker']} {m['year']} page {m['page_num']}" for m in image_metadatas]

for i in range(0, len(image_ids), CHROMA_ADD_BATCH_SIZE):
    end = i + CHROMA_ADD_BATCH_SIZE
    image_collection.add(
        ids=image_ids[i:end],
        embeddings=image_embeddings[i:end],
        documents=image_documents[i:end],
        metadatas=image_metadatas[i:end],
    )

print(f"Embedded and stored {len(image_ids)} images in ChromaDB 'image_chunks' collection.")

In [ ]:
print(f"kept_images: {len(kept_images)}")
print(f"Vectors actually stored in ChromaDB image_chunks: {image_collection.count()}")
if len(kept_images) != image_collection.count():
    print("  WARNING: counts don't match -- some images may have failed to embed/store.")
else:
    print("  OK: all kept images are stored.")

if kept_images:
    from collections import Counter
    label_counts = Counter(img["label"] for img in kept_images)
    print("\nBreakdown by label:")
    for label, count in label_counts.most_common():
        print(f"  {label}: {count}")

    doc_counts = Counter((img["ticker"], img["year"]) for img in kept_images)
    print("\nBreakdown by document:")
    for (ticker, year), count in sorted(doc_counts.items()):
        print(f"  {ticker} {year}: {count}")

In [ ]:
# Run one sanity-check query against each index (BM25, Chroma text/image)

print("=" * 60)
print("SANITY CHECK 1: BM25 (term-based)")
print("=" * 60)
query_tokens = simple_tokenize("Nvidia data center revenue")
scores = bm25_index.get_scores(query_tokens)
top_idx = scores.argsort()[-3:][::-1]
for i in top_idx:
    c = all_chunks[i]
    print(f"  [{c['ticker']} {c['year']}] {c['text'][:100]}...")

print()
print("=" * 60)
print("SANITY CHECK 2: ChromaDB (semantic, text)")
print("=" * 60)
query_embedding = voyage_client.multimodal_embed(
    inputs=[["how did the company describe supply chain risk"]],
    model="voyage-multimodal-3", input_type="query"
).embeddings[0]
results = text_collection.query(query_embeddings=[query_embedding], n_results=3)
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"  [{meta['ticker']} {meta['year']} / {meta['section']}] {doc[:100]}...")

print()
print("=" * 60)
print("SANITY CHECK 3: ChromaDB (semantic, images)")
print("=" * 60)
print(f"  Total images stored: {image_collection.count()}")

print()
print("Both indexes populated. Ready for retrieval_pipeline.ipynb.")